# Student Grade Predictor Using Machine Learning

**Minor Internship Project | B.Tech CSE (AI Specialization)**

This notebook predicts a student's final score using attendance, assignment performance, previous exam score, study hours, and internal assessment score.

> **Dataset note:** The dataset is synthetically generated inside this notebook. It is intended for educational demonstration and does not represent real student records.

## 2. Project Overview

The project applies **Multiple Linear Regression** to estimate a student's final score from academic and study-related features.

The workflow is:
1. Generate a realistic synthetic dataset.
2. Explore and visualize the data.
3. Prepare features and target.
4. Split the data into training and testing sets.
5. Train a Linear Regression model.
6. Evaluate the model using MAE, MSE, RMSE and R².
7. Predict a new student's score interactively.
8. Convert the predicted score into a project-defined grade.

This follows the repository's beginner-level Student Grade Predictor idea, while adding more educational features for a complete internship submission.

## 3. Problem Statement

Students' final academic performance can be influenced by factors such as attendance, assignment performance, previous examination results, study time, and internal assessment.

The objective is to build a simple machine-learning model that learns relationships between these inputs and a final score, then uses the learned relationships to estimate the final score of a new student.

## 4. Objectives

- Build a beginner-friendly machine-learning project.
- Generate at least 500 realistic synthetic student records.
- Analyze relationships among academic features.
- Train a Linear Regression model.
- Evaluate prediction quality using standard regression metrics.
- Provide an interactive student prediction section.
- Convert predicted scores into simple project-defined grades.

## 5. Technologies Used

- **Python** – programming language
- **Pandas** – data handling
- **NumPy** – numerical operations and synthetic data generation
- **Matplotlib** – plotting
- **Seaborn** – statistical visualization
- **Scikit-learn** – train-test split, Linear Regression and evaluation metrics
- **Google Colab** – execution environment

## 6. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproducibility and display settings
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

## 7. Dataset Generation

In [ ]:
# Generate a synthetic dataset of 600 students.
# A fixed random seed makes the generated data reproducible.

rng = np.random.default_rng(RANDOM_STATE)
N_STUDENTS = 600

attendance = np.clip(rng.normal(78, 11, N_STUDENTS), 45, 100)
assignment_score = np.clip(rng.normal(72, 14, N_STUDENTS), 35, 100)
previous_exam_score = np.clip(rng.normal(68, 15, N_STUDENTS), 30, 100)
study_hours = np.clip(rng.normal(3.8, 1.7, N_STUDENTS), 0.5, 9)
internal_assessment = np.clip(rng.normal(70, 13, N_STUDENTS), 35, 100)

# The target is influenced by all five features.
# A small interaction term and random noise make the relationship
# realistic rather than perfectly linear.
noise = rng.normal(0, 3.5, N_STUDENTS)

final_score = (
    0.20 * attendance
    + 0.25 * assignment_score
    + 0.22 * previous_exam_score
    + 2.2 * study_hours
    + 0.23 * internal_assessment
    + 0.012 * (attendance - 75) * (assignment_score - 70)
    + 0.10 * np.sqrt(study_hours) * previous_exam_score / 10
    + noise
)

# Keep final scores in the sensible 0–100 range.
final_score = np.clip(final_score, 0, 100)

df = pd.DataFrame({
    "attendance": np.round(attendance, 1),
    "assignment_score": np.round(assignment_score, 1),
    "previous_exam_score": np.round(previous_exam_score, 1),
    "study_hours": np.round(study_hours, 1),
    "internal_assessment": np.round(internal_assessment, 1),
    "final_score": np.round(final_score, 1)
})

print(f"Dataset created successfully with {len(df)} student records.")
df.head()

## 8. Dataset Description

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn information:")
df.info()

print("\nStatistical summary:")
display(df.describe().round(2))

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## 9. Exploratory Data Analysis

In [ ]:
# Check the ranges of the main variables.
summary = pd.DataFrame({
    "Minimum": df.min(),
    "Maximum": df.max(),
    "Mean": df.mean(),
    "Median": df.median()
}).round(2)

display(summary)

print("The target variable is final_score.")
print("The five input features are attendance, assignment_score, previous_exam_score, study_hours, and internal_assessment.")

## 10. Data Visualization

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["final_score"], bins=20, kde=True, color="steelblue")
plt.title("Distribution of Final Scores")
plt.xlabel("Final Score")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="attendance", y="final_score", alpha=0.65)
plt.title("Attendance vs Final Score")
plt.xlabel("Attendance (%)")
plt.ylabel("Final Score")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="study_hours", y="final_score", alpha=0.65)
plt.title("Study Hours vs Final Score")
plt.xlabel("Study Hours per Day")
plt.ylabel("Final Score")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="assignment_score", y="final_score", alpha=0.65)
plt.title("Assignment Score vs Final Score")
plt.xlabel("Assignment Score")
plt.ylabel("Final Score")
plt.show()

## 11. Feature Selection

In [ ]:
FEATURES = [
    "attendance",
    "assignment_score",
    "previous_exam_score",
    "study_hours",
    "internal_assessment"
]
TARGET = "final_score"

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("Selected features:")
for feature in FEATURES:
    print("-", feature)

print("\nTarget:", TARGET)

## 12. Data Preprocessing

In [ ]:
# Basic preprocessing checks.
# The synthetic dataset is already numeric and has no missing values,
# so no categorical encoding or imputation is required.

X = X.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(y, errors="coerce")

# Remove any accidental invalid rows if they ever occur.
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

# Keep feature values within their expected project ranges.
X["attendance"] = X["attendance"].clip(0, 100)
X["assignment_score"] = X["assignment_score"].clip(0, 100)
X["previous_exam_score"] = X["previous_exam_score"].clip(0, 100)
X["study_hours"] = X["study_hours"].clip(0, 24)
X["internal_assessment"] = X["internal_assessment"].clip(0, 100)
y = y.clip(0, 100)

print("Preprocessing completed.")
print("Rows available:", len(X))
print("Missing feature values:", int(X.isnull().sum().sum()))
print("Missing target values:", int(y.isnull().sum()))

## 13. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training percentage:", round(len(X_train) / len(X) * 100, 1), "%")
print("Testing percentage:", round(len(X_test) / len(X) * 100, 1), "%")

## 14. Linear Regression Model

In [ ]:
model = LinearRegression()
print("Linear Regression model created successfully.")

## 15. Model Training

In [ ]:
model.fit(X_train, y_train)

print("Model training completed.")
print("\nIntercept:", round(model.intercept_, 3))

coefficients = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": model.coef_
}).sort_values("Coefficient", key=lambda s: s.abs(), ascending=False)

display(coefficients.round(3))

## 16. Model Prediction

In [ ]:
y_pred = model.predict(X_test)

# Keep displayed predictions within the valid score range.
y_pred_clipped = np.clip(y_pred, 0, 100)

prediction_table = pd.DataFrame({
    "Actual Score": y_test.values,
    "Predicted Score": np.round(y_pred_clipped, 2)
})

display(prediction_table.head(10))

## 17. Model Evaluation

In [ ]:
mae = mean_absolute_error(y_test, y_pred_clipped)
mse = mean_squared_error(y_test, y_pred_clipped)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_clipped)

metrics_df = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r2]
})

display(metrics_df.round(4))

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

print("\nInterpretation:")
print(f"On average, the model's prediction differs from the actual score by about {mae:.2f} marks.")
print("R² indicates how much of the variation in final scores is explained by the model on the test set.")

## 18. Actual vs Predicted Visualization

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred_clipped, alpha=0.7)

line_min = min(y_test.min(), y_pred_clipped.min())
line_max = max(y_test.max(), y_pred_clipped.max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle="--")

plt.title("Actual vs Predicted Final Scores")
plt.xlabel("Actual Final Score")
plt.ylabel("Predicted Final Score")
plt.show()

## 19. Residual / Error Analysis

In [ ]:
residuals = y_test.values - y_pred_clipped

plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred_clipped, y=residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.title("Residual/Error Plot")
plt.xlabel("Predicted Final Score")
plt.ylabel("Residual (Actual - Predicted)")
plt.show()

print(f"Mean residual: {residuals.mean():.2f}")
print("Residuals close to zero indicate smaller prediction errors.")

## 20. Interactive Student Prediction

In [ ]:
def get_validated_float(prompt, minimum, maximum):
    """Keep asking until the user enters a valid number in the given range."""
    while True:
        try:
            value = float(input(prompt))
            if not np.isfinite(value):
                print("Invalid input. Please enter a finite number.")
                continue
            if value < minimum or value > maximum:
                print(f"Please enter a value from {minimum} to {maximum}.")
                continue
            return value
        except (ValueError, TypeError):
            print("Invalid input. Please enter a numeric value.")


def grade_from_score(score):
    """Convert a 0–100 score into the project's defined grade."""
    if score >= 90:
        return "A+"
    elif score >= 80:
        return "A"
    elif score >= 70:
        return "B"
    elif score >= 60:
        return "C"
    elif score >= 50:
        return "D"
    else:
        return "F"


def performance_interpretation(score):
    """Give a simple non-institutional interpretation."""
    if score >= 80:
        return "Predicted performance is strong."
    elif score >= 60:
        return "Predicted performance is satisfactory."
    elif score >= 50:
        return "Predicted performance is moderate; improvement may help."
    else:
        return "Predicted performance is below the project's passing threshold."

In [ ]:
print("Enter student details.")
print("Allowed ranges: percentages/scores = 0–100; study hours = 0–24.\n")

new_student = {
    "attendance": get_validated_float("Attendance percentage (0–100): ", 0, 100),
    "assignment_score": get_validated_float("Assignment average (0–100): ", 0, 100),
    "previous_exam_score": get_validated_float("Previous exam score (0–100): ", 0, 100),
    "study_hours": get_validated_float("Study hours per day (0–24): ", 0, 24),
    "internal_assessment": get_validated_float("Internal assessment score (0–100): ", 0, 100)
}

new_student_df = pd.DataFrame([new_student], columns=FEATURES)

raw_prediction = float(model.predict(new_student_df)[0])
predicted_score = float(np.clip(raw_prediction, 0, 100))
predicted_grade = grade_from_score(predicted_score)

print("\n" + "=" * 45)
print("STUDENT GRADE PREDICTION")
print("=" * 45)
print(f"Predicted Final Score : {predicted_score:.2f}/100")
print(f"Predicted Grade       : {predicted_grade}")
print(f"Interpretation        : {performance_interpretation(predicted_score)}")
print("=" * 45)

Enter student details.
Allowed ranges: percentages/scores = 0–100; study hours = 0–24.



## 21. Grade Classification

In [ ]:
# Project-defined grading thresholds.
# These are examples only and should be changed according to an institution's policy.

grade_boundaries = pd.DataFrame({
    "Score Range": ["90–100", "80–89.99", "70–79.99", "60–69.99", "50–59.99", "Below 50"],
    "Grade": ["A+", "A", "B", "C", "D", "F"]
})

display(grade_boundaries)

print("Important: These grading thresholds are project-defined examples.")
print("They are not an official institutional grading policy.")

## 22. Example Predictions

In [ ]:
example_students = pd.DataFrame([
    {
        "attendance": 92,
        "assignment_score": 88,
        "previous_exam_score": 85,
        "study_hours": 6,
        "internal_assessment": 90
    },
    {
        "attendance": 76,
        "assignment_score": 70,
        "previous_exam_score": 68,
        "study_hours": 4,
        "internal_assessment": 72
    },
    {
        "attendance": 58,
        "assignment_score": 52,
        "previous_exam_score": 48,
        "study_hours": 2,
        "internal_assessment": 55
    }
], columns=FEATURES)

example_predictions = np.clip(model.predict(example_students), 0, 100)

example_results = example_students.copy()
example_results["predicted_final_score"] = np.round(example_predictions, 2)
example_results["grade"] = [grade_from_score(score) for score in example_predictions]

display(example_results)

## 23. Conclusion

This project demonstrates how Linear Regression can be used to estimate student final scores from multiple academic and study-related inputs. The model is trained and tested using a reproducible synthetic dataset, and its performance is measured with MAE, MSE, RMSE and R².

The interactive section allows a user to enter student information and obtain a predicted final score and project-defined grade.

Because the dataset is synthetic and the model is intentionally simple, the results should be treated as an educational demonstration rather than a real academic decision-making system.

## 24. Future Scope

- Replace synthetic data with a properly collected real-world dataset.
- Add more relevant academic and behavioral features.
- Compare Linear Regression with other suitable algorithms.
- Add cross-validation and hyperparameter experiments.
- Build a web interface using a suitable application framework.
- Add database storage for authorized academic use.
- Include model monitoring and fairness checks before real deployment.
- Adapt grade thresholds to an institution's official grading policy.